# Weakly-supervised crack segmentation

Segmenting cracks pixel by pixel while training on one bit per image: does this image contain a
crack or not.

Dataset: [Crack Segmentation Dataset](https://www.kaggle.com/datasets/lakshaymiddha/crack-segmentation-dataset), only the `train/` directory is used.

Jakub Laskowski (160287), Jakub Górniak (160326)

## What we are allowed to look at

| Stage | What it sees |
|---|---|
| Label creation | Each mask is opened once and turned into one bit: is there any crack pixel in this image. The mask is then thrown away. |
| Training (all stages) | Images and that one bit. |
| Evaluation | The real masks, on a held out split, after everything is frozen. |

No threshold and no hyperparameter is picked by looking at a mask.

## Why this is not a crack detector

Nothing here knows what a crack looks like. There are no edge detectors, no ridge filters, no
morphological thinning and no colour rules. We only assume that

1. a positive image has at least one positive pixel and a negative image has none,
2. the thing we are looking for covers a small part of the image,
3. its boundaries line up with edges in the image.

The same three things are true for "find the tumour cells given only healthy/sick slide labels", so
changing `DATA_ROOT` is enough to move the notebook to that problem.

## The three stages

```
       image + 1 bit                    coarse evidence                 full-resolution mask
    +----------------+              +--------------------+            +--------------------+
    |  1. MIL        |   score map  |  2. CAM -> pseudo  |  pseudo-   |  3. U-Net          |
    |  classifier    |------------->|     mask           |----------->|  distillation      |
    |  (ResNet-34)   |   40x40      |  TTA + guided      |  masks     |  (ResNet-34 enc.)  |
    +----------------+              |  filter + dual thr |            +--------------------+
                                    +--------------------+
```

1. MIL classifier. A fully convolutional ResNet-34 outputs a per pixel logit map and top-k pooling
   turns it into one logit for the whole image. Training only needs the bit, but the map in the
   middle is already a class activation map.
2. CAM to pseudo-mask. Multi-scale and flip TTA, a guided filter to sharpen the edges, then two
   thresholds: sure foreground, sure background, and an ignore band in between that we drop from the
   next loss.
3. Distillation. A U-Net is trained on the pseudo-masks. It averages out the CAM noise and gives
   full resolution masks. Then one round of self-training where the U-Net relabels the data, always
   forcing negative images to be completely background.

In [ ]:
import math
import os
import random
import re
import time
import warnings
from pathlib import Path

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from albumentations.pytorch import ToTensorV2
from PIL import Image
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# --------------------------------------------------------------------------- config
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/kaggle/input/crack-segmentation-dataset/crack_segmentation_dataset"))
TRAIN_DIR = DATA_ROOT / "train"          # the only directory we are allowed to train on
WORK_DIR = Path(os.environ.get("WORK_DIR", "/kaggle/working"))

# these are the values behind the reported results, the env overrides are there so we could
# run the whole notebook quickly as a smoke test
IMG_SIZE = int(os.environ.get("IMG_SIZE", 320))          # classifier/segmenter input resolution
BATCH_SIZE = int(os.environ.get("BATCH_SIZE", 16))
CLS_EPOCHS = int(os.environ.get("CLS_EPOCHS", 8))        # stage 1 - MIL classifier
SEG_EPOCHS = int(os.environ.get("SEG_EPOCHS", 10))       # stage 3 - U-Net, per self-training round
SELF_TRAINING_ROUNDS = int(os.environ.get("SELF_TRAINING_ROUNDS", 2))
SEED = 42

TOPK_RATIO = 0.02       # "the object covers ~2% of the image" prior, used by the MIL pooling
IGNORE_INDEX = 255      # pixels the pseudo-labeller is unsure about

FG_THRESHOLD = 0.45     # dual threshold on the normalised CAM
BG_THRESHOLD = 0.20
TTA_SCALES = (0.75, 1.0, 1.5, 2.0)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def sample_n(frame, n, seed=SEED):
    """Sample at most n rows so the figures still work on a small subset."""
    return frame.sample(min(n, len(frame)), random_state=seed)


set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE} | torch {torch.__version__}")
print(f"data:   {TRAIN_DIR} (exists: {TRAIN_DIR.exists()})")


## 1. Data preparation, throwing the masks away

The cell below is the only place where a training mask is opened. Each one becomes
`label = int(mask.any())` and we cache the result in a CSV with just `image_path, label`. Everything
after that reads the CSV, so pixel information cannot leak into the training by accident.

The split is stratified on the label and on the source sub-dataset (the filename prefix, `CFD_`,
`DeepCrack_`, `Volker_` and so on). Those sub-datasets look very different, so a plain random split
would give us a validation set that is too easy.

In [ ]:
LABEL_CSV = WORK_DIR / "image_labels.csv"
WORK_DIR.mkdir(parents=True, exist_ok=True)


def source_group(stem: str) -> str:
    """Filename prefix, 'CFD_006' -> 'cfd'."""
    match = re.match(r"^([A-Za-z]+)", stem)
    return match.group(1).lower() if match else "unknown"


def build_label_table() -> pd.DataFrame:
    """Open every mask once, keep one bit per image and forget the pixels."""
    if LABEL_CSV.exists():
        return pd.read_csv(LABEL_CSV)

    image_paths = sorted((TRAIN_DIR / "images").glob("*"))
    mask_dir = TRAIN_DIR / "masks"
    rows = []
    for path in tqdm(image_paths, desc="deriving image-level labels"):
        mask_path = next(mask_dir.glob(path.stem + ".*"))
        mask = np.asarray(Image.open(mask_path).convert("L"))
        rows.append(
            {
                "image_path": str(path),
                "stem": path.stem,
                "group": source_group(path.stem),
                "label": int((mask > 127).any()),   # the one bit we keep
            }
        )
    table = pd.DataFrame(rows)
    table.to_csv(LABEL_CSV, index=False)
    return table


table = build_label_table()

# stratify on (label, source) so every split sees every imaging condition
strata = table.label.astype(str) + "|" + table.group
strata = strata.where(strata.map(strata.value_counts()) >= 3, "rare|" + table.label.astype(str))

train_idx, holdout_idx = train_test_split(np.arange(len(table)), test_size=0.25, random_state=SEED, stratify=strata)
val_idx, test_idx = train_test_split(holdout_idx, test_size=0.6, random_state=SEED, stratify=strata.iloc[holdout_idx])

table["split"] = "train"
table.loc[val_idx, "split"] = "val"
table.loc[test_idx, "split"] = "test"

train_df = table[table.split == "train"].reset_index(drop=True)
val_df = table[table.split == "val"].reset_index(drop=True)
test_df = table[table.split == "test"].reset_index(drop=True)

print(f"total images: {len(table)}")
print(table.groupby("split").agg(n=("label", "size"), positives=("label", "sum")))


In [ ]:
# EDA on what we are allowed to see: the images and their one bit.
counts = table.groupby(["group", "label"]).size().unstack(fill_value=0)
counts = counts.loc[counts.sum(axis=1).sort_values(ascending=False).index]

fig = plt.figure(figsize=(14, 4.5))
gs = fig.add_gridspec(1, 2, width_ratios=[1.3, 1])

ax = fig.add_subplot(gs[0])
counts.plot(kind="bar", stacked=True, ax=ax, color=["#9ecae1", "#e6550d"], width=0.8)
ax.set_title("Images per source sub-dataset")
ax.set_xlabel("")
ax.set_ylabel("images")
ax.legend(["no crack", "crack"])
ax.tick_params(axis="x", rotation=45)

ax = fig.add_subplot(gs[1])
overall = table.label.value_counts().sort_index()
ax.bar(["no crack", "crack"], overall.values, color=["#9ecae1", "#e6550d"])
for i, v in enumerate(overall.values):
    ax.text(i, v, f"{v}\n({v / len(table):.0%})", ha="center", va="bottom")
ax.set_title("Overall class balance")
ax.set_ylim(0, overall.max() * 1.2)
plt.tight_layout()
plt.show()

sizes = {Image.open(p).size for p in sample_n(table, 50).image_path}
print(f"image sizes in a random sample: {sizes}")

sample = pd.concat([sample_n(table[table.label == 1], 6), sample_n(table[table.label == 0], 6)])
fig, axes = plt.subplots(2, 6, figsize=(15, 5.2))
for ax, (_, row) in zip(axes.ravel(), sample.iterrows(), strict=False):
    ax.imshow(Image.open(row.image_path))
    ax.set_title(f"{'crack' if row.label else 'no crack'} · {row.group}", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
fig.suptitle("What the model gets to see during training", y=1.0)
plt.tight_layout()
plt.show()


### Preprocessing and augmentation

Resize to 320x320, ImageNet normalisation, and augmentations that do not change the label and do not
depend on the orientation: flips and 90 degree rotations (a crack upside down is still a crack),
small affine warps, and brightness/blur/noise so the classifier cannot just learn the exposure of
one source sub-dataset.

`CrackDataset` returns the image and the label and never stores a mask path, so the rule is enforced
by the code and not by us remembering it.

In [ ]:
train_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Affine(scale=(0.8, 1.25), rotate=(-20, 20), p=0.5),
    A.RandomBrightnessContrast(0.25, 0.25, p=0.5),
    A.HueSaturationValue(8, 20, 10, p=0.3),
    A.OneOf([A.GaussianBlur(blur_limit=(3, 5)), A.MotionBlur(blur_limit=5)], p=0.2),
    A.GaussNoise(p=0.2),
    A.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ToTensorV2(),
])

eval_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ToTensorV2(),
])


def denormalise(tensor):
    """CHW normalised tensor -> HWC image in [0, 1], for plotting."""
    array = tensor.detach().cpu().numpy().transpose(1, 2, 0)
    return np.clip(array * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN), 0, 1)


class CrackDataset(Dataset):
    """Images and one binary label, there is no mask anywhere in here."""

    def __init__(self, frame, transform, with_raw=False):
        self.paths = frame.image_path.tolist()
        self.labels = frame.label.astype(np.float32).tolist()
        self.stems = frame.stem.tolist()
        self.transform = transform
        self.with_raw = with_raw

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        raw = np.asarray(Image.open(self.paths[idx]).convert("RGB"))
        item = {"image": self.transform(image=raw)["image"], "label": self.labels[idx], "stem": self.stems[idx]}
        if self.with_raw:
            item["raw"] = cv2.resize(raw, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        return item


def make_loader(frame, transform, shuffle=False, batch_size=BATCH_SIZE, with_raw=False):
    return DataLoader(
        CrackDataset(frame, transform, with_raw),
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=2,
        pin_memory=torch.cuda.is_available(),
        drop_last=shuffle,
    )


train_loader = make_loader(train_df, train_tf, shuffle=True)
val_loader = make_loader(val_df, eval_tf)
print(f"train batches: {len(train_loader)} | val batches: {len(val_loader)}")


## 2. Stage 1, a classifier that also gives us a map

### Picking the architecture

The image is a bag and every position of the feature map is an instance. The label says either "at
least one instance is positive" or "all of them are negative", which is multiple instance learning.
It is the same setup as a slide that is positive if it contains at least one tumour cell.

So we build the network as a segmenter and only collapse it to one number at the end:

```
image -> ResNet-34 (ImageNet, dilated) -> 3x3 conv + BN + ReLU -> 1x1 conv -> score map S (B,1,40,40)
                                                                                  |
                                                                            top-k pooling
                                                                                  v
                                                                            image logit (B,1)
```

| Choice | Alternative | Why |
|---|---|---|
| Dilated ResNet-34 (no stride in `layer3`/`layer4`, dilation 2 and 4, output stride 8) | plain ResNet-50 at stride 32 | Cracks are a few pixels wide. Stride 32 gives a 10x10 map which is useless. Dilation gives 40x40 with the same parameters and the same receptive field, and ResNet-34 keeps the 4x compute increase manageable. `replace_stride_with_dilation` only works on `Bottleneck` nets so we had to do it by hand. |
| Top-k pooling, k = 2% of the positions | global average pooling | GAP averages 1600 positions, so a 30 pixel crack is about 2% of the logit and the gradient drowns. Top-k is a soft maximum and its k says "the object is small". Global max pooling is the other extreme, correct in principle but a single position gives a very noisy gradient. |
| No bias on the 1x1 conv | with a bias | Without it the image logit stays a pooled linear projection of the features, which is what makes S an actual CAM and not an approximation of one. |

There is also an auxiliary max pooling head on the same score map. It says that on a negative image
no position is allowed to fire, which is how the negative images give us real supervision for the
background.

In [ ]:
def dilate_stage(stage, dilation):
    """Turn a stride-2 ResNet stage into a stride-1 dilated stage (keeps the receptive field)."""
    for module in stage.modules():
        if isinstance(module, nn.Conv2d):
            if module.stride == (2, 2):
                module.stride = (1, 1)
            if module.kernel_size == (3, 3):
                module.dilation = (dilation, dilation)
                module.padding = (dilation, dilation)


class MILClassifier(nn.Module):
    """Fully-convolutional ResNet-34 + 1x1 classifier + top-k MIL pooling."""

    def __init__(self, pretrained=True, topk_ratio=TOPK_RATIO):
        super().__init__()
        weights = torchvision.models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = torchvision.models.resnet34(weights=weights)
        # torchvision only does replace_stride_with_dilation for Bottleneck nets,
        # so we do it by hand. Output stride goes from 32 to 8.
        dilate_stage(backbone.layer3, 2)
        dilate_stage(backbone.layer4, 4)

        self.encoder = nn.Sequential(*list(backbone.children())[:-2])   # -> (B, 512, H/8, W/8)
        self.neck = nn.Sequential(
            nn.Conv2d(512, 256, 3, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.1),
        )
        self.classifier = nn.Conv2d(256, 1, 1, bias=False)   # no bias, otherwise it is not a real CAM
        self.topk_ratio = topk_ratio

    def score_map(self, x):
        """Per-pixel logits at input_size / 8."""
        return self.classifier(self.neck(self.encoder(x)))

    def topk_pool(self, score):
        flat = score.flatten(2)                                   # (B, 1, H*W)
        k = max(1, round(self.topk_ratio * flat.shape[-1]))
        return flat.topk(k, dim=-1).values.mean(-1)               # (B, 1)

    @staticmethod
    def max_pool(score):
        return score.flatten(2).max(-1).values                    # (B, 1)

    def forward(self, x):
        score = self.score_map(x)
        return {"logit": self.topk_pool(score), "max_logit": self.max_pool(score), "score_map": score}


def normalise_map(maps, eps=1e-6):
    """Per-image min-max scaling of an activation map to [0, 1]."""
    flat = maps.flatten(2)
    lo = flat.min(-1, keepdim=True).values
    hi = flat.max(-1, keepdim=True).values
    return ((flat - lo) / (hi - lo + eps)).view_as(maps).clamp(0, 1)


model = MILClassifier().to(DEVICE)
with torch.no_grad():
    out = model(torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=DEVICE))
print(f"score map: {tuple(out['score_map'].shape)}   image logit: {tuple(out['logit'].shape)}")
print(f"trainable parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"top-k pooling uses k = {max(1, round(TOPK_RATIO * out['score_map'][0, 0].numel()))} of {out['score_map'][0, 0].numel()} positions")


### The loss: one bit plus self-consistency

$$\mathcal{L} = \underbrace{\mathrm{BCE}(\mathrm{topk}(S), y)}_{\text{bag}} + 0.4\,\underbrace{\mathrm{BCE}(\max(S), y)}_{\text{strict MIL}} + \underbrace{\mathrm{BCE}(\mathrm{topk}(\tilde S), y) + \lambda_p \lVert S - \tilde S \rVert_1}_{\text{puzzle consistency}} + \lambda_s\,\overline{\sigma(S)}\big|_{y=1}$$

The usual problem with CAM based segmentation is that a classifier only needs the most obvious part
of the object to get the label right, so the map lights up a small blob and stops there. Two extra
terms deal with that and neither of them needs more than the bit we already have.

* Puzzle consistency (Puzzle-CAM). Cut the image into 2x2 tiles, classify each tile on its own and
  stitch the four maps back together into $\tilde S$. A tile cannot use evidence that lives in
  another tile, so the network has to respond everywhere the evidence is, and $\tilde S$ has to
  agree with the whole image map $S$. This was the single most useful term we tried.
* Foreground area prior ($\lambda_s = 0.02$, positives only), a small push towards a smaller
  activated area. This is assumption 2, set it to 0 if the target is large.

The regularisers are warmed up linearly over the first two epochs. Before that $S$ is basically
noise and forcing it to be consistent with itself only slows the training down.

In [ ]:
W_MAX, W_PUZZLE, W_SPARSITY = 0.4, 0.5, 0.02
REG_WARMUP_EPOCHS = 2


def tile(x, n=2):
    """(B,C,H,W) -> (B*n*n, C, H/n, W/n), row-major tile order."""
    b, c, h, w = x.shape
    x = x.reshape(b, c, n, h // n, n, w // n).permute(0, 2, 4, 1, 3, 5)
    return x.reshape(b * n * n, c, h // n, w // n)


def untile(x, batch, n=2):
    """Inverse of `tile`."""
    _, c, th, tw = x.shape
    x = x.reshape(batch, n, n, c, th, tw).permute(0, 3, 1, 4, 2, 5)
    return x.reshape(batch, c, n * th, n * tw)


def mil_loss(model, images, labels, reg_scale=1.0):
    out = model(images)
    terms = {
        "bag": F.binary_cross_entropy_with_logits(out["logit"].squeeze(1), labels),
        "strict": W_MAX * F.binary_cross_entropy_with_logits(out["max_logit"].squeeze(1), labels),
    }

    if reg_scale > 0 and W_PUZZLE > 0:
        tiled = untile(model.score_map(tile(images)), images.shape[0])
        terms["puzzle_cls"] = F.binary_cross_entropy_with_logits(model.topk_pool(tiled).squeeze(1), labels)
        terms["puzzle_l1"] = reg_scale * W_PUZZLE * F.l1_loss(out["score_map"], tiled)

    if reg_scale > 0 and W_SPARSITY > 0 and (labels > 0.5).any():
        terms["sparsity"] = reg_scale * W_SPARSITY * torch.sigmoid(out["score_map"][labels > 0.5]).mean()

    total = torch.stack(list(terms.values())).sum()
    return total, {k: float(v.detach()) for k, v in terms.items()} | {"total": float(total.detach())}


In [ ]:
CLS_CKPT = WORK_DIR / "mil_classifier.pth"


def cosine_with_warmup(optimizer, total_steps, warmup=200):
    def lr_lambda(step):
        if step < warmup:
            return (step + 1) / warmup
        progress = (step - warmup) / max(total_steps - warmup, 1)
        return 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


@torch.no_grad()
def classifier_scores(model, loader):
    """Image level probabilities and labels, no masks involved."""
    model.eval()
    scores, labels = [], []
    for batch in loader:
        logit = model(batch["image"].to(DEVICE))["logit"].squeeze(1)
        scores.append(torch.sigmoid(logit).float().cpu().numpy())
        labels.append(batch["label"].numpy())
    return np.concatenate(scores), np.concatenate(labels)


def train_classifier(model, epochs=CLS_EPOCHS):
    head_params = [p for n, p in model.named_parameters() if not n.startswith("encoder")]
    optimizer = torch.optim.AdamW(
        [{"params": model.encoder.parameters(), "lr": 3e-5}, {"params": head_params, "lr": 3e-4}],
        weight_decay=1e-4,
    )
    scheduler = cosine_with_warmup(optimizer, epochs * len(train_loader))
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    history, best_auc = [], -1.0

    for epoch in range(epochs):
        model.train()
        reg_scale = min(1.0, epoch / REG_WARMUP_EPOCHS) if REG_WARMUP_EPOCHS else 1.0
        running, started = {}, time.time()

        for batch in tqdm(train_loader, desc=f"epoch {epoch + 1}/{epochs}", leave=False):
            images = batch["image"].to(DEVICE, non_blocking=True)
            labels = batch["label"].to(DEVICE, non_blocking=True)
            with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
                loss, report = mil_loss(model, images, labels, reg_scale)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            for k, v in report.items():
                running[k] = running.get(k, 0.0) + v

        train_stats = {k: v / len(train_loader) for k, v in running.items()}
        scores, labels = classifier_scores(model, val_loader)
        val_auc = roc_auc_score(labels, scores)
        val_acc = ((scores >= 0.5) == labels).mean()
        history.append({"epoch": epoch + 1, **train_stats, "val_auroc": val_auc, "val_acc": val_acc})
        print(f"epoch {epoch + 1:2d}  loss {train_stats['total']:.4f}  val AUROC {val_auc:.4f}  val acc {val_acc:.4f}  ({time.time() - started:.0f}s)")

        if val_auc > best_auc:                      # selection uses image-level labels only
            best_auc = val_auc
            torch.save(model.state_dict(), CLS_CKPT)
    return pd.DataFrame(history)


set_seed()
cls_history = train_classifier(model)
model.load_state_dict(torch.load(CLS_CKPT, map_location=DEVICE))
print(f"best val AUROC: {cls_history.val_auroc.max():.4f}")


In [ ]:
test_loader = make_loader(test_df, eval_tf)
test_scores, test_labels = classifier_scores(model, test_loader)
test_pred = (test_scores >= 0.5).astype(int)

tp = int(((test_pred == 1) & (test_labels == 1)).sum())
fp = int(((test_pred == 1) & (test_labels == 0)).sum())
fn = int(((test_pred == 0) & (test_labels == 1)).sum())
tn = int(((test_pred == 0) & (test_labels == 0)).sum())
precision, recall = tp / max(tp + fp, 1), tp / max(tp + fn, 1)

cls_metrics = {
    "accuracy": (tp + tn) / len(test_labels),
    "precision": precision,
    "recall": recall,
    "f1": 2 * precision * recall / max(precision + recall, 1e-9),
    "auroc": roc_auc_score(test_labels, test_scores),
    "average_precision": average_precision_score(test_labels, test_scores),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
loss_cols = [c for c in cls_history.columns if c not in {"epoch", "val_auroc", "val_acc", "total"}]
for col in loss_cols:
    axes[0].plot(cls_history.epoch, cls_history[col], label=col, alpha=0.8)
axes[0].plot(cls_history.epoch, cls_history.total, "k--", lw=2, label="total")
axes[0].set_title("Training loss terms")
axes[0].set_xlabel("epoch")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].plot(cls_history.epoch, cls_history.val_auroc, "o-", label="val AUROC")
axes[1].plot(cls_history.epoch, cls_history.val_acc, "s-", label="val accuracy")
axes[1].set_title("Image-level validation")
axes[1].set_xlabel("epoch")
axes[1].set_ylim(0.5, 1.02)
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].imshow([[tn, fp], [fn, tp]], cmap="Blues")
for (i, j), v in np.ndenumerate([[tn, fp], [fn, tp]]):
    axes[2].text(j, i, str(v), ha="center", va="center", fontsize=14)
axes[2].set_xticks([0, 1], ["pred no crack", "pred crack"])
axes[2].set_yticks([0, 1], ["no crack", "crack"])
axes[2].set_title("Test confusion matrix")
plt.tight_layout()
plt.show()

print("image-level test metrics:")
for k, v in cls_metrics.items():
    print(f"  {k:18s} {v:.4f}")


## 3. Stage 2, from a 40x40 score map to pixels

Three generic steps, in this order:

1. Multi-scale and flip TTA. We recompute the score map at scales 0.75 / 1.0 / 1.5 / 2.0 with
   horizontal and vertical flips and average the 16 passes. The small scales give context and the
   2.0 scale gives resolution (at 640 px input the map is effectively 80x80). Averaging also cancels
   part of the orientation bias that a stack of convolutions always has.
2. Guided filter. The CAM is smooth and limited by the stride, but the image itself already knows
   where the edges are, so a guided filter (He et al.) copies that structure onto the probability
   map. This is assumption 3 and it is not task specific, it is the same filter you would use to
   snap a tumour heat map to the cell boundaries.
3. Two thresholds. Instead of deciding everywhere, pixels above `fg` become foreground, pixels below
   `bg` become background and everything in between becomes `ignore` and is dropped from the stage 3
   loss. Letting the pseudo-labeller say "I don't know" is what stops the U-Net from copying the
   CAM's mistakes. The foreground threshold is mixed 50/50 with a per image Otsu threshold so images
   with weak and strong evidence get treated differently. Otsu is computed from the CAM histogram,
   never from a mask.

On top of that there is the image level gate: if the classifier says there is no crack, the whole
pseudo-mask is background. That is the only exact supervision we have and we apply it at every
stage, self-training included.

In [ ]:
@torch.no_grad()
def multi_scale_cam(model, images, scales=TTA_SCALES, flips=True, size=(IMG_SIZE, IMG_SIZE)):
    """Scale/flip-averaged score map, min-max normalised per image to [0, 1]."""
    model.eval()
    accumulator = torch.zeros(images.shape[0], 1, *size, device=images.device)
    variants = [(), (-1,), (-2,)] if flips else [()]

    for scale in scales:
        scaled = images if scale == 1.0 else F.interpolate(
            images, scale_factor=scale, mode="bilinear", align_corners=False)
        for dims in variants:
            batch = torch.flip(scaled, dims) if dims else scaled
            score = model.score_map(batch)
            if dims:
                score = torch.flip(score, dims)
            accumulator += F.interpolate(F.relu(score), size=size, mode="bilinear", align_corners=False)

    return normalise_map(accumulator / (len(scales) * len(variants)))


def guided_filter(guide_rgb, source, radius=8, eps=1e-3):
    """Edge preserving filter, pulls source onto the edges of guide_rgb."""
    guide = cv2.cvtColor(guide_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    source = source.astype(np.float32)
    ksize = (2 * radius + 1, 2 * radius + 1)

    def box(x):
        return cv2.boxFilter(x, -1, ksize, normalize=True, borderType=cv2.BORDER_REFLECT)

    mean_g, mean_s = box(guide), box(source)
    var_g = box(guide * guide) - mean_g * mean_g
    cov_gs = box(guide * source) - mean_g * mean_s
    a = cov_gs / (var_g + eps)
    b = mean_s - a * mean_g
    return np.clip(box(a) * guide + box(b), 0, 1)


def rescale(x, eps=1e-6):
    lo, hi = float(x.min()), float(x.max())
    return (x - lo) / (hi - lo + eps)


def otsu_threshold(probability, bins=256):
    """Otsu's threshold, computed from the CAM histogram alone."""
    histogram, edges = np.histogram(probability.ravel(), bins=bins, range=(0.0, 1.0))
    probability_mass = histogram.astype(np.float64) / max(histogram.sum(), 1)
    centres = (edges[:-1] + edges[1:]) / 2
    omega = np.cumsum(probability_mass)
    mu = np.cumsum(probability_mass * centres)
    with np.errstate(divide="ignore", invalid="ignore"):
        between = (mu[-1] * omega - mu) ** 2 / (omega * (1 - omega))
    between[~np.isfinite(between)] = -1
    return float(centres[int(np.argmax(between))])


def probability_to_pseudo_label(probability, is_positive):
    """0 = background, 1 = foreground, IGNORE_INDEX = unknown."""
    if not is_positive:
        return np.zeros(probability.shape, np.uint8)          # no crack in the image, so no crack pixels
    fg = 0.5 * otsu_threshold(probability) + 0.5 * FG_THRESHOLD
    bg = min(BG_THRESHOLD, 0.75 * fg)
    label = np.full(probability.shape, IGNORE_INDEX, np.uint8)
    label[probability < bg] = 0
    label[probability >= fg] = 1
    return label


def refined_probability(cam_slice, raw_image):
    """CAM -> refined, renormalised probability map at IMG_SIZE."""
    return rescale(guided_filter(raw_image, cam_slice))


In [ ]:
def load_gt_mask(stem, size=None):
    """Ground truth, only for the plots and the scoring, never for training."""
    path = next((TRAIN_DIR / "masks").glob(stem + ".*"))
    mask = (np.asarray(Image.open(path).convert("L")) > 127).astype(np.uint8)
    return cv2.resize(mask, size, interpolation=cv2.INTER_NEAREST) if size else mask


# walk a few positive test images through every step of stage 2
demo_df = sample_n(test_df[test_df.label == 1], 4).reset_index(drop=True)
demo_batch = next(iter(make_loader(demo_df, eval_tf, batch_size=len(demo_df), with_raw=True)))
demo_images = demo_batch["image"].to(DEVICE)

single_cam = multi_scale_cam(model, demo_images, scales=(1.0,), flips=False)[:, 0].cpu().numpy()
tta_cam = multi_scale_cam(model, demo_images)[:, 0].cpu().numpy()

titles = ["image", "CAM (single scale)", "CAM + TTA", "+ guided filter", "pseudo-mask", "ground truth"]
fig, axes = plt.subplots(len(demo_df), 6, figsize=(17, 2.9 * len(demo_df)), squeeze=False)
for i in range(len(demo_df)):
    raw = demo_batch["raw"][i].numpy()
    refined = refined_probability(tta_cam[i], raw)
    pseudo = probability_to_pseudo_label(refined, is_positive=True)
    truth = load_gt_mask(demo_batch["stem"][i], (IMG_SIZE, IMG_SIZE))

    for j, panel in enumerate([raw, single_cam[i], tta_cam[i], refined, pseudo, truth]):
        ax = axes[i][j]
        if j == 0:
            ax.imshow(panel)
        elif j == 4:
            ax.imshow(np.where(panel == IGNORE_INDEX, 0.5, panel), cmap="gray", vmin=0, vmax=1)
        elif j == 5:
            ax.imshow(panel, cmap="gray", vmin=0, vmax=1)
        else:
            ax.imshow(panel, cmap="inferno", vmin=0, vmax=1)
        if i == 0:
            ax.set_title(titles[j], fontsize=11)
        ax.axis("off")
fig.text(0.63, 0.005, "pseudo-mask: black = background, grey = ignore, white = foreground", ha="center", fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
PSEUDO_DIR = WORK_DIR / "pseudo"


@torch.no_grad()
def generate_pseudo_masks(model, frame, out_dir):
    """Write one uint8 PNG per image: {0 background, 1 foreground, 255 ignore}."""
    out_dir.mkdir(parents=True, exist_ok=True)
    loader = make_loader(frame, eval_tf, with_raw=True)
    composition = {"foreground": 0.0, "background": 0.0, "ignore": 0.0}

    for batch in tqdm(loader, desc=f"pseudo-labelling -> {out_dir.name}", leave=False):
        cams = multi_scale_cam(model, batch["image"].to(DEVICE))[:, 0].cpu().numpy()
        for i, stem in enumerate(batch["stem"]):
            refined = refined_probability(cams[i], batch["raw"][i].numpy())
            # train/val images come with their image-level bit, so the gate is exact here
            label = probability_to_pseudo_label(refined, bool(batch["label"][i] > 0.5))
            Image.fromarray(label).save(out_dir / f"{stem}.png")
            composition["foreground"] += float((label == 1).mean())
            composition["background"] += float((label == 0).mean())
            composition["ignore"] += float((label == IGNORE_INDEX).mean())

    return {k: v / len(frame) for k, v in composition.items()}


set_seed()
pseudo_stats = {split: generate_pseudo_masks(model, frame, PSEUDO_DIR / split)
                for split, frame in [("train", train_df), ("val", val_df)]}

print("average pseudo-mask composition")
print(pd.DataFrame(pseudo_stats).T.round(4))


## 4. Stage 3, distilling the pseudo-masks into a U-Net

A CAM is a low resolution detector, a U-Net with skip connections is what actually draws the
outline. Training the U-Net on the confident parts of the pseudo-masks gives us three things:

* Resolution. The skip connections carry the stride 2 and stride 4 detail that the classifier's
  stride 8 trunk had thrown away, so the output is really full resolution.
* Denoising. The CAM errors are mostly independent between images, so a network that has to explain
  thousands of them with one set of weights fits the part that is consistent and averages the rest
  away. The student ends up better than the teacher.
* Room to disagree. The ignore band means 20-40% of the pixels have no target at all, so the network
  is not punished for correcting the teacher where the teacher was unsure.

Loss: partial BCE (`pos_weight = 4` because of the foreground/background imbalance) plus a soft Dice
term, both only on the pixels that are not ignored.

Self-training: after round 1 the U-Net relabels the training set itself, `p >= 0.7` is foreground,
`p <= 0.3` is background, the rest is ignore, and negative images are forced to background. Round 2
trains on those. Model selection uses the validation loss on the pseudo-masks, so still no real
masks.

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )

    def forward(self, x, skip=None):
        x = F.interpolate(x, scale_factor=2.0, mode="nearest")
        if skip is not None:
            x = torch.cat([x, skip], dim=1)
        return self.block(x)


class UNet(nn.Module):
    """Classic U-Net with an ImageNet-pretrained ResNet-34 encoder."""

    def __init__(self, pretrained=True):
        super().__init__()
        weights = torchvision.models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = torchvision.models.resnet34(weights=weights)
        self.stem = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu)   # /2,  64
        self.pool, self.layer1 = backbone.maxpool, backbone.layer1               # /4,  64
        self.layer2, self.layer3, self.layer4 = backbone.layer2, backbone.layer3, backbone.layer4
        self.dec4 = DecoderBlock(512, 256, 256)
        self.dec3 = DecoderBlock(256, 128, 128)
        self.dec2 = DecoderBlock(128, 64, 64)
        self.dec1 = DecoderBlock(64, 64, 32)
        self.dec0 = DecoderBlock(32, 0, 16)
        self.head = nn.Conv2d(16, 1, 3, padding=1)

    def forward(self, x):
        f0 = self.stem(x)
        f1 = self.layer1(self.pool(f0))
        f2 = self.layer2(f1)
        f3 = self.layer3(f2)
        f4 = self.layer4(f3)
        d = self.dec4(f4, f3)
        d = self.dec3(d, f2)
        d = self.dec2(d, f1)
        d = self.dec1(d, f0)
        return self.head(self.dec0(d))


class PseudoMaskDataset(Dataset):
    """Image plus the mask we generated ourselves."""

    def __init__(self, frame, pseudo_dir, transform):
        self.paths = frame.image_path.tolist()
        self.stems = frame.stem.tolist()
        self.pseudo_dir = Path(pseudo_dir)
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        image = np.asarray(Image.open(self.paths[idx]).convert("RGB"))
        pseudo = np.asarray(Image.open(self.pseudo_dir / f"{self.stems[idx]}.png"))
        # pseudo-masks live at IMG_SIZE, the images at their native size
        image = cv2.resize(image, (pseudo.shape[1], pseudo.shape[0]), interpolation=cv2.INTER_LINEAR)
        augmented = self.transform(image=image, mask=pseudo)
        return {"image": augmented["image"], "target": augmented["mask"].long()}


# only flips and rotations, an affine warp would have to invent border pixels and we
# would not know whether to mark them as ignore
seg_train_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(0.25, 0.25, p=0.5),
    A.HueSaturationValue(8, 20, 10, p=0.3),
    A.GaussNoise(p=0.2),
    A.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ToTensorV2(),
])
seg_eval_tf = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.Normalize(IMAGENET_MEAN, IMAGENET_STD), ToTensorV2()])


def pseudo_loss(logits, target, pos_weight=4.0):
    """Partial BCE + soft Dice, evaluated only where the pseudo-labeller was confident."""
    logits = logits.squeeze(1)
    valid = target != IGNORE_INDEX
    if valid.sum() == 0:
        return logits.sum() * 0.0
    weight = torch.tensor(pos_weight, device=logits.device)
    ce = F.binary_cross_entropy_with_logits(logits[valid], target[valid].float(), pos_weight=weight)

    probability = torch.sigmoid(logits) * valid
    reference = (target == 1).float()
    intersection = (probability * reference).sum((1, 2))
    dice = 1 - ((2 * intersection + 1) / (probability.sum((1, 2)) + reference.sum((1, 2)) + 1)).mean()
    return ce + dice


print(f"U-Net parameters: {sum(p.numel() for p in UNet(pretrained=False).parameters()) / 1e6:.1f}M")


In [ ]:
SEG_CKPT = WORK_DIR / "unet.pth"
CONFIDENCE = 0.7


def make_seg_loader(frame, pseudo_dir, transform, shuffle):
    return DataLoader(
        PseudoMaskDataset(frame, pseudo_dir, transform),
        batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=2,
        pin_memory=torch.cuda.is_available(), drop_last=shuffle,
    )


def run_seg_epoch(unet, loader, optimizer=None, scheduler=None, scaler=None):
    training = optimizer is not None
    unet.train(training)
    total = 0.0
    for batch in tqdm(loader, desc="train" if training else "val", leave=False):
        images = batch["image"].to(DEVICE, non_blocking=True)
        targets = batch["target"].to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(training), torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
            loss = pseudo_loss(unet(images), targets)
        if training:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(unet.parameters(), 5.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
        total += float(loss.detach())
    return total / len(loader)


@torch.no_grad()
def relabel_with_unet(unet, frame, out_dir):
    """Self-training, the U-Net relabels the data. The image level gate still applies."""
    out_dir.mkdir(parents=True, exist_ok=True)
    unet.eval()
    for batch in tqdm(make_loader(frame, eval_tf), desc=f"re-labelling -> {out_dir.name}", leave=False):
        probability = torch.sigmoid(unet(batch["image"].to(DEVICE))).squeeze(1).float().cpu().numpy()
        for i, stem in enumerate(batch["stem"]):
            if batch["label"][i] < 0.5:
                label = np.zeros(probability[i].shape, np.uint8)     # negative image, always fully background
            else:
                label = np.full(probability[i].shape, IGNORE_INDEX, np.uint8)
                label[probability[i] <= 1 - CONFIDENCE] = 0
                label[probability[i] >= CONFIDENCE] = 1
            Image.fromarray(label).save(out_dir / f"{stem}.png")


set_seed()
unet = UNet().to(DEVICE)
seg_history, pseudo_dir = [], PSEUDO_DIR

for round_id in range(1, SELF_TRAINING_ROUNDS + 1):
    train_seg_loader = make_seg_loader(train_df, pseudo_dir / "train", seg_train_tf, shuffle=True)
    val_seg_loader = make_seg_loader(val_df, pseudo_dir / "val", seg_eval_tf, shuffle=False)

    encoder_params = [p for n, p in unet.named_parameters() if n.startswith(("stem", "layer"))]
    decoder_params = [p for n, p in unet.named_parameters() if not n.startswith(("stem", "layer"))]
    optimizer = torch.optim.AdamW(
        [{"params": encoder_params, "lr": 3e-5}, {"params": decoder_params, "lr": 3e-4}], weight_decay=1e-4)
    scheduler = cosine_with_warmup(optimizer, SEG_EPOCHS * len(train_seg_loader), warmup=100)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    best_val = float("inf")

    for epoch in range(1, SEG_EPOCHS + 1):
        started = time.time()
        train_loss = run_seg_epoch(unet, train_seg_loader, optimizer, scheduler, scaler)
        val_loss = run_seg_epoch(unet, val_seg_loader)
        seg_history.append({"round": round_id, "epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
        print(f"round {round_id} epoch {epoch:2d}  train {train_loss:.4f}  val {val_loss:.4f}  ({time.time() - started:.0f}s)")
        if val_loss < best_val:                     # selection on pseudo-masks, not ground truth
            best_val = val_loss
            torch.save(unet.state_dict(), SEG_CKPT)

    unet.load_state_dict(torch.load(SEG_CKPT, map_location=DEVICE))
    if round_id < SELF_TRAINING_ROUNDS:
        pseudo_dir = WORK_DIR / f"pseudo_round{round_id + 1}"
        for split, frame in [("train", train_df), ("val", val_df)]:
            relabel_with_unet(unet, frame, pseudo_dir / split)

seg_history = pd.DataFrame(seg_history)


## 5. Results

Everything above is frozen now. The test split was never seen by any training loop, any threshold or
any model selection decision, so we can finally open its masks.

Every variant gives a probability map at 320x320 which we upsample to the mask's own resolution
before thresholding, so all of them are scored on the same pixels. The gate at test time uses the
classifier's prediction and not the true bit, because a deployed model would not have the bit
either.

| Metric | What it tells us |
|---|---|
| IoU / Dice (over the whole dataset) | Overall pixel agreement, pooled over all images. |
| Mean image IoU | The same but averaged per image over the cracked ones, so a few huge cracks do not dominate. |
| Precision / recall | Whether we over or under segment. |
| Boundary F1 (+-2 px) | Region metrics are dominated by area, which is harsh for something a few pixels wide. This asks whether the contour is roughly in the right place. |

The four variants below separate the contribution of each idea: the raw CAM, the TTA, the guided
filter, and the U-Net distillation with self-training.

In [ ]:
def boundary_f1(prediction, target, tolerance=2):
    """F1 between predicted and true contours, allowing a few pixels of slack."""
    if prediction.sum() == 0 and target.sum() == 0:
        return 1.0
    if prediction.sum() == 0 or target.sum() == 0:
        return 0.0
    kernel = np.ones((3, 3), np.uint8)
    slack = np.ones((2 * tolerance + 1, 2 * tolerance + 1), np.uint8)
    pred_edge = cv2.morphologyEx(prediction, cv2.MORPH_GRADIENT, kernel)
    true_edge = cv2.morphologyEx(target, cv2.MORPH_GRADIENT, kernel)
    precision = (pred_edge * cv2.dilate(true_edge, slack)).sum() / max(pred_edge.sum(), 1)
    recall = (true_edge * cv2.dilate(pred_edge, slack)).sum() / max(true_edge.sum(), 1)
    return float(2 * precision * recall / max(precision + recall, 1e-9))


@torch.no_grad()
def predict_probability(variant, batch):
    """Probability map at IMG_SIZE + the image-level gate from the classifier."""
    images = batch["image"].to(DEVICE)
    gate = (torch.sigmoid(model(images)["logit"].squeeze(1)).cpu().numpy() >= 0.5)

    if variant == "cam":
        probability = multi_scale_cam(model, images, scales=(1.0,), flips=False)[:, 0].cpu().numpy()
    elif variant == "cam_tta":
        probability = multi_scale_cam(model, images)[:, 0].cpu().numpy()
    elif variant == "cam_tta_gf":
        cams = multi_scale_cam(model, images)[:, 0].cpu().numpy()
        probability = np.stack([refined_probability(cams[i], batch["raw"][i].numpy()) for i in range(len(cams))])
    elif variant == "unet":
        probability = torch.sigmoid(unet(images)).squeeze(1).float().cpu().numpy()
    else:
        raise ValueError(variant)
    return probability, gate


def binarise(probability, variant):
    """Same rule as the pseudo-labeller. For the U-Net plain 0.5 works."""
    if variant == "unet":
        return (probability >= 0.5).astype(np.uint8)
    threshold = 0.5 * otsu_threshold(probability) + 0.5 * FG_THRESHOLD
    return (probability >= threshold).astype(np.uint8)


def evaluate(variant, frame=None, tag=None):
    frame = test_df if frame is None else frame
    loader = make_loader(frame, eval_tf, with_raw=True)
    tp = fp = fn = tn = 0
    image_iou, boundary = [], []

    for batch in tqdm(loader, desc=f"evaluating {variant}", leave=False):
        probability, gate = predict_probability(variant, batch)
        for i, stem in enumerate(batch["stem"]):
            truth = load_gt_mask(stem)
            prob = cv2.resize(probability[i], (truth.shape[1], truth.shape[0]), interpolation=cv2.INTER_LINEAR)
            prediction = binarise(prob, variant) if gate[i] else np.zeros_like(truth)

            p, t = prediction.astype(bool), truth.astype(bool)
            i_tp, i_fp, i_fn = int((p & t).sum()), int((p & ~t).sum()), int((~p & t).sum())
            tp, fp, fn, tn = tp + i_tp, fp + i_fp, fn + i_fn, tn + int((~p & ~t).sum())
            if t.any():
                image_iou.append(i_tp / max(i_tp + i_fp + i_fn, 1))
                boundary.append(boundary_f1(prediction, truth))

    precision, recall = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return {
        "variant": tag or variant,
        "IoU": tp / max(tp + fp + fn, 1),
        "Dice": 2 * tp / max(2 * tp + fp + fn, 1),
        "precision": precision,
        "recall": recall,
        "mean image IoU": float(np.mean(image_iou)) if image_iou else 0.0,
        "boundary F1": float(np.mean(boundary)) if boundary else 0.0,
        "pixel acc": (tp + tn) / max(tp + tn + fp + fn, 1),
    }


set_seed()
results = pd.DataFrame([
    evaluate("cam", tag="1. CAM (single scale)"),
    evaluate("cam_tta", tag="2. + multi-scale/flip TTA"),
    evaluate("cam_tta_gf", tag="3. + guided-filter refinement"),
    evaluate("unet", tag=f"4. + U-Net distillation ({SELF_TRAINING_ROUNDS} rounds)"),
]).set_index("variant")

display(results.round(4))


In [ ]:
seg_history = pd.DataFrame(seg_history)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

x = np.arange(len(results))
for offset, metric, colour in [(-0.2, "IoU", "#3182bd"), (0.0, "Dice", "#e6550d"), (0.2, "boundary F1", "#31a354")]:
    axes[0].bar(x + offset, results[metric], width=0.2, label=metric, color=colour)
axes[0].set_xticks(x, [f"{i + 1}" for i in range(len(results))])
axes[0].set_xlabel("ablation step")
axes[0].set_title("Ablation on the test split")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

axes[1].plot(results.recall, results.precision, "o-", color="#756bb1")
for name, row in results.iterrows():
    axes[1].annotate(name.split(".")[0], (row.recall, row.precision), textcoords="offset points", xytext=(6, 4))
axes[1].set_xlabel("recall")
axes[1].set_ylabel("precision")
axes[1].set_title("Precision / recall trade-off")
axes[1].grid(alpha=0.3)

for round_id, group in seg_history.groupby("round"):
    steps = np.arange(len(group)) + (round_id - 1) * SEG_EPOCHS
    axes[2].plot(steps, group.train_loss, label=f"round {round_id} train")
    axes[2].plot(steps, group.val_loss, "--", label=f"round {round_id} val")
axes[2].set_xlabel("epoch")
axes[2].set_title("U-Net loss on pseudo-masks")
axes[2].legend(fontsize=8)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(results[["IoU", "Dice", "boundary F1"]].round(4).to_string())


In [ ]:
@torch.no_grad()
def show_predictions(frame, title):
    frame = frame.reset_index(drop=True)
    batch = next(iter(make_loader(frame, eval_tf, batch_size=len(frame), with_raw=True)))
    images = batch["image"].to(DEVICE)
    cam = multi_scale_cam(model, images)[:, 0].cpu().numpy()
    unet_prob = torch.sigmoid(unet(images)).squeeze(1).float().cpu().numpy()

    columns = ["image", "CAM (stage 2)", "U-Net probability", "U-Net mask", "ground truth", "error map"]
    fig, axes = plt.subplots(len(frame), 6, figsize=(17, 2.85 * len(frame)), squeeze=False)
    for i, stem in enumerate(batch["stem"]):
        truth = load_gt_mask(stem, (IMG_SIZE, IMG_SIZE))
        prediction = (unet_prob[i] >= 0.5).astype(np.uint8)
        error = np.stack([prediction & (1 - truth), prediction & truth, truth & (1 - prediction)], -1).astype(float)

        for j, panel in enumerate([batch["raw"][i].numpy(), cam[i], unet_prob[i], prediction, truth, error]):
            ax = axes[i][j]
            if j in (1, 2):
                ax.imshow(panel, cmap="inferno", vmin=0, vmax=1)
            elif j in (3, 4):
                ax.imshow(panel, cmap="gray", vmin=0, vmax=1)
            else:
                ax.imshow(panel)
            if i == 0:
                ax.set_title(columns[j], fontsize=11)
            ax.axis("off")
    fig.suptitle(f"{title}   ·   error map: green = correct, red = false positive, blue = missed", y=1.005)
    plt.tight_layout()
    plt.show()


positives = test_df[test_df.label == 1]
show_predictions(sample_n(positives, 5), "Typical predictions")

# failure analysis: rank a sample of cracked images by how badly the U-Net disagrees with the truth
scan = sample_n(positives, 200)
scores = []
for _, row in tqdm(scan.iterrows(), total=len(scan), desc="scanning for failures"):
    image = eval_tf(image=np.asarray(Image.open(row.image_path).convert("RGB")))["image"][None].to(DEVICE)
    with torch.no_grad():
        prediction = (torch.sigmoid(unet(image))[0, 0].float().cpu().numpy() >= 0.5).astype(np.uint8)
    truth = load_gt_mask(row.stem, (IMG_SIZE, IMG_SIZE))
    scores.append((int((prediction & truth).sum()) / max(int((prediction | truth).sum()), 1), row.stem))

worst = [stem for _, stem in sorted(scores)[:5]]
show_predictions(test_df[test_df.stem.isin(worst)], f"Worst {len(worst)} of {len(scan)} sampled cracked images")


In [ ]:
# The sweep below is an oracle curve, picking its best point would use the test masks. It is
# only here to show how much we lose by using our mask-free 0.5 instead.
thresholds = np.linspace(0.05, 0.95, 19)
sweep_tp = np.zeros_like(thresholds)
sweep_fp = np.zeros_like(thresholds)
sweep_fn = np.zeros_like(thresholds)
per_source = {}

sample_df = test_df.sample(min(400, len(test_df)), random_state=SEED).reset_index(drop=True)
for batch in tqdm(make_loader(sample_df, eval_tf, with_raw=True), desc="threshold sweep", leave=False):
    probability, gate = predict_probability("unet", batch)
    for i, stem in enumerate(batch["stem"]):
        truth = load_gt_mask(stem, (IMG_SIZE, IMG_SIZE)).astype(bool)
        prob = probability[i] * gate[i]
        for k, threshold in enumerate(thresholds):
            prediction = prob >= threshold
            hit = int((prediction & truth).sum())
            sweep_tp[k] += hit
            sweep_fp[k] += int(prediction.sum()) - hit
            sweep_fn[k] += int(truth.sum()) - hit
        if truth.any():
            prediction = prob >= 0.5
            hit = int((prediction & truth).sum())
            group = stem.split("_")[0].lower()
            per_source.setdefault(group, []).append(hit / max(int((prediction | truth).sum()), 1))

sweep_iou = sweep_tp / np.maximum(sweep_tp + sweep_fp + sweep_fn, 1)
source_iou = pd.Series({k: np.mean(v) for k, v in per_source.items()}).sort_values()
source_n = pd.Series({k: len(v) for k, v in per_source.items()})

fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
axes[0].plot(thresholds, sweep_iou, "o-")
axes[0].axvline(0.5, color="crimson", ls="--", label="threshold we actually use (mask-free)")
axes[0].scatter([thresholds[sweep_iou.argmax()]], [sweep_iou.max()], color="green", zorder=5,
                label=f"oracle best = {sweep_iou.max():.3f} @ {thresholds[sweep_iou.argmax()]:.2f}")
axes[0].set_xlabel("binarisation threshold")
axes[0].set_ylabel("dataset IoU")
axes[0].set_title("How much does the threshold matter?")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].barh(source_iou.index, source_iou.values, color="#3182bd")
for i, (name, value) in enumerate(source_iou.items()):
    axes[1].text(value, i, f"  {value:.2f}  (n={source_n[name]})", va="center", fontsize=8)
axes[1].set_xlim(0, min(1.0, source_iou.max() * 1.45))
axes[1].set_xlabel("mean image IoU")
axes[1].set_title("Where does it work? (per source sub-dataset)")
plt.tight_layout()
plt.show()


## 6. Analysis and conclusions

### What each step gave us

Going down the ablation table, one change per row:

1. Single scale CAM is the baseline. The location is right, the boundaries are not. At stride 8 a
   3 pixel crack is a fraction of one cell.
2. Multi-scale and flip TTA costs no extra training, only around 16x the inference time, and the 2.0
   scale is what recovers the thin structure.
3. The guided filter is where the map stops looking like a blob. It is also the step where most
   people would put a domain specific edge detector, and using a generic one is what keeps the
   pipeline reusable.
4. U-Net distillation with self-training is the biggest single jump, and the student ends up better
   than the teacher it was trained on. The CAM errors are inconsistent between images while the real
   signal is not, and the ignore band leaves the student room to correct the teacher.

### Where it fails

* Thin low contrast cracks on textured concrete. The classifier can be right for the wrong reason
  (the texture) and then the CAM covers a region instead of a line.
* Shadows, joints and painted lines look the same as a crack at the image level. They are most of
  our false positives and an image level label cannot tell them apart.
* Cracks touching the border are under segmented. The puzzle term helps but the border tiles still
  see the least context.
* Recall is higher than precision everywhere. With only a bag level label the model learns where the
  evidence is, and the evidence is thicker than the crack itself. That is the weakly supervised gap,
  not something we can tune away.

### Compared to full supervision

A fully supervised U-Net on this dataset gets around 0.65-0.75 IoU. Getting a good part of that with
no pixel annotations at all is the point of the exercise, and the annotation cost is one click per
image instead of a traced mask.

### Using it on something else

Nothing above is crack specific. To run it on "find the tumour cells given only healthy/sick slide
labels":

1. Point `DATA_ROOT` at the new dataset and give it `image_path, label`.
2. Set `TOPK_RATIO` to a rough guess of how much of the image the target covers, and
   `W_SPARSITY = 0` if the target is large.
3. Nothing else changes.

### What we would try next

* A learned pixel affinity network (IRNet / AffinityNet) instead of the guided filter, to spread the
  CAM seeds along learned boundaries instead of image gradients.
* Contrastive scale equivariance (SEAM) next to the puzzle consistency.
* Training the classifier at output stride 4 with a bigger crop. The stride 8 map is still the
  tightest bottleneck in the whole pipeline.